# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show basic description
print(f"Dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This gives an understanding of the main table-like datasets in Croissant and their fields.

In [ ]:
# Explore record sets and fields via the dataset metadata

record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f" - RecordSet name: {rs.name} (ID: {rs.id})")
    # List the fields of this record set
    print("   Fields:")
    for field in rs.fields:
        print(f"     - {field.name} (ID: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as discovered above.

In [ ]:
# Load all record sets into dataframes and display their columns

dataframes = dict()

for rs in record_sets:
    print(f"Loading: {rs.name} (ID: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}\n")

# Pick main record set for further analysis (most datasets have one main set, if in doubt use the first)
main_rs_id = record_sets[0].id if record_sets else None
df = dataframes[main_rs_id] if main_rs_id else None
print(f"Using record_set '{main_rs_id}' for EDA. Showing first rows:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filtering, normalization, and group analysis.

For this demo we'll pick a numeric field (e.g. GAD-7 score, PHQ-9 score, or PSQ score if present). All field references are by `@id` names.

In [ ]:
# Find a numeric field
numeric_fields = [f.id for f in record_sets[0].fields if f.data_type in ('Integer', 'Float', 'Number')]
print(f"Numeric fields: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]  # use the first numeric field by @id
    print(f"Analyzing field: {numeric_field}")
    threshold = df[numeric_field].mean() if numeric_field in df.columns else 10
    # Simple filter: rows above the mean
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered to {len(filtered_df)} records with {numeric_field} > {threshold:.2f}")
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find a group field (typically demographic, e.g. by 'village', 'level_of_education', etc.)
    # Heuristically pick any categorical string column
    non_numeric_fields = [f.id for f in record_sets[0].fields if f.data_type == 'Text' or f.data_type == 'String']
    if non_numeric_fields:
        group_field = non_numeric_fields[0]
        print(f"Grouping by: {group_field}")
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
else:
    print("No numeric fields detected.")

## 5. Visualization
Visualize the numeric field distribution and the group means if detected.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if df is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field in df.columns:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(8,4))
        sns.barplot(y=group_means.index, x=group_means.values)
        plt.xlabel(f"Mean {numeric_field}")
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.show()

## 6. Conclusion
We have demonstrated how to load, overview, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. You can adapt the above workflow for your own analysis, exploring additional fields and creating further visualizations as needed.

**Key takeaways:**
- All field and record set references are by `@id` for reproducibility.
- You can access all metadata using `dataset.metadata` for further programmatic exploration.
- The dataset demonstrates rich, schema-aware metadata via Croissant, enabling rapid AI/ML analysis workflows.
